In [1]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_rows",100)
pd.set_option("display.max_columns",100)

## Combine the data from all quarters from 2017 to 2019

In [2]:
for year in range(2017,2020):
    for quarter in range(1,5):    
        fname = "PUDF_base1_" + str(quarter) + "q" + str(year) + "_cleaned.csv"
        print(fname)
        df_tmp = pd.read_csv(fname, low_memory=False)

        df_tmp['SOURCE_OF_ADMISSION'] = df_tmp['SOURCE_OF_ADMISSION'].astype(str) # Remove rows with SOURCE_OF_ADMISSION = "`" (invalid value)
        df_tmp.drop(df_tmp[df_tmp["SOURCE_OF_ADMISSION"]=="`"].index,inplace=True)

        if quarter==1 and year==2017:
            df = df_tmp
        else:
            df = pd.concat([df, df_tmp], ignore_index=True) 
        del df_tmp

df.reset_index(drop=True, inplace=True)


PUDF_base1_1q2017_cleaned.csv
PUDF_base1_2q2017_cleaned.csv
PUDF_base1_3q2017_cleaned.csv
PUDF_base1_4q2017_cleaned.csv
PUDF_base1_1q2018_cleaned.csv
PUDF_base1_2q2018_cleaned.csv
PUDF_base1_3q2018_cleaned.csv
PUDF_base1_4q2018_cleaned.csv
PUDF_base1_1q2019_cleaned.csv
PUDF_base1_2q2019_cleaned.csv
PUDF_base1_3q2019_cleaned.csv
PUDF_base1_4q2019_cleaned.csv


## Cleaning the data

We set the data types of the categorical data.
We also introduced two additional variables.

In [3]:
df["TYPE_OF_ADMISSION"] = df["TYPE_OF_ADMISSION"].astype(int).astype(str)
df["PUBLIC_HEALTH_REGION"] = df["PUBLIC_HEALTH_REGION"].astype(int).astype(str)
df["RACE"] = df["RACE"].astype(int).astype(str)
df["ETHNICITY"] = df["ETHNICITY"].astype(int).astype(str)
df["ADMIT_WEEKDAY"] = df["ADMIT_WEEKDAY"].astype(int).astype(str)

df["PROLONGED"] = [1 if los >= 30 else 0 for los in df["LENGTH_OF_STAY"]] # Create prolonged stay indicator variable
df["NUM_CODES"] = df[["CODE_"+str(n) for n in range(1,22)]].sum(axis=1) # Count number of diagnosis codes for each patient

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7006583 entries, 0 to 7006582
Data columns (total 47 columns):
 #   Column                Dtype  
---  ------                -----  
 0   RECORD_ID             int64  
 1   THCIC_ID              int64  
 2   TYPE_OF_ADMISSION     object 
 3   SOURCE_OF_ADMISSION   object 
 4   PUBLIC_HEALTH_REGION  object 
 5   PAT_STATUS            int64  
 6   SEX_CODE              object 
 7   RACE                  object 
 8   ETHNICITY             object 
 9   ADMIT_WEEKDAY         object 
 10  LENGTH_OF_STAY        float64
 11  PAT_AGE               int64  
 12  EMERGENCY_DEPT_FLAG   object 
 13  DIAG_CODES_OA         object 
 14  CODE_1                int64  
 15  CODE_2                int64  
 16  CODE_3                int64  
 17  CODE_4                int64  
 18  CODE_5                int64  
 19  CODE_6                int64  
 20  CODE_7                int64  
 21  CODE_8                int64  
 22  CODE_9                int64  
 23  CODE_10

Remove feature columns not needed for training -- RECORD_ID, THCIC_ID, PAT_ZIP, PAT_STATE, PAT_COUNTRY, PAT_COUNTY, DIAG_CODES_OA, PROVIDER_ZIP, CODE_22, PAT_STATUS

In [4]:
redundant_features = ["RECORD_ID", "THCIC_ID", "PAT_ZIP", "PAT_STATE", "PAT_COUNTRY","PAT_COUNTY", "DIAG_CODES_OA", "PROVIDER_ZIP", "CODE_22","PAT_STATUS"]

for feature in redundant_features:
    if feature in df.columns:
        df.drop(columns=[feature], inplace=True)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7006583 entries, 0 to 7006582
Data columns (total 42 columns):
 #   Column                Dtype  
---  ------                -----  
 0   TYPE_OF_ADMISSION     object 
 1   SOURCE_OF_ADMISSION   object 
 2   PUBLIC_HEALTH_REGION  object 
 3   SEX_CODE              object 
 4   RACE                  object 
 5   ETHNICITY             object 
 6   ADMIT_WEEKDAY         object 
 7   LENGTH_OF_STAY        float64
 8   PAT_AGE               int64  
 9   EMERGENCY_DEPT_FLAG   object 
 10  CODE_1                int64  
 11  CODE_2                int64  
 12  CODE_3                int64  
 13  CODE_4                int64  
 14  CODE_5                int64  
 15  CODE_6                int64  
 16  CODE_7                int64  
 17  CODE_8                int64  
 18  CODE_9                int64  
 19  CODE_10               int64  
 20  CODE_11               int64  
 21  CODE_12               int64  
 22  CODE_13               int64  
 23  CODE_14

Merge two lowest patient age groups

In [5]:
df.loc[df["PAT_AGE"]==0,"PAT_AGE"] = 1

In [6]:
df["EMERGENCY_DEPT_FLAG"] = df["EMERGENCY_DEPT_FLAG"].map({"Y":1,"N":0})

## Create pipeline for preprocessing and training

In [7]:
categorical_cols = ["TYPE_OF_ADMISSION", "SOURCE_OF_ADMISSION", "PUBLIC_HEALTH_REGION", "SEX_CODE", "RACE", "ETHNICITY"]

Split data for training and testing

In [8]:
# Import Metrics
from sklearn.metrics import mean_absolute_error 
from sklearn.metrics import root_mean_squared_error 
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold

df['STRATA'] = df['QUARTER'].astype(str) + '_' + df['PAT_RURAL'].astype(str) + '_' + df['PROVIDER_RURAL'].astype(str) # for stratification sampling

# df_small = df.sample(n=2000000, random_state=42)
df_small = df

y = df_small["LENGTH_OF_STAY"]
X = df_small.drop(columns=["LENGTH_OF_STAY","PROLONGED","STRATA"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify = df_small['STRATA']
)

In [9]:
# Create the StratifiedKFold
strata_train = df.loc[X_train.index, "STRATA"]

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=2222
)

In [16]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
# from cuml.ensemble import RandomForestRegressor

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols)
    ],
    remainder="passthrough"  # Keeps the 'age' column untouched
)

model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    # ('rf_gpu', RandomForestRegressor(n_estimators=100))
    ("regressor", RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=64)) # Use 64 processing cores for parallel processing
])


In [17]:
mae_scores = []
rmse_scores = []

for fold, (train_idx, valid_idx) in enumerate(skf.split(X_train, strata_train)):
    X_tr = X_train.iloc[train_idx].copy()
    X_val = X_train.iloc[valid_idx].copy()
    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[valid_idx]

    model_pipeline.fit(X_tr, y_tr)
    pred_y = model_pipeline.predict(X_val)

    mae_score = mean_absolute_error(y_val, pred_y)
    rmse_score = root_mean_squared_error(y_val, pred_y)

    mae_scores.append(mae_score)
    rmse_scores.append(rmse_score)

    print(f"Fold {fold + 1}: MAE = {mae_score}, RMSE = {rmse_score}")

Fold 1: MAE = 3.2656801238951276, RMSE = 13.500672318148652
Fold 2: MAE = 3.2775978608606464, RMSE = 14.939717666147065
Fold 3: MAE = 3.2644101920606072, RMSE = 14.746789955760738
Fold 4: MAE = 3.2668004382212636, RMSE = 15.165399399332316
Fold 5: MAE = 3.2537516806165554, RMSE = 13.616565764849073
